# Identifying and Removing Duplicate Records

This milestone focuses on **identifying and removing duplicate records** in Pandas DataFrames.

Duplicate data is a common data quality issue that can **skew analysis**, inflate counts, and lead to incorrect conclusions if not handled properly.

**Dataset:** Air Quality readings across major Indian cities (PM2.5, PM10, NO2, CO, AQI, Temperature, Humidity)

---

### What You Will Learn
| Goal | Method |
|------|--------|
| Detect all duplicate rows | `df.duplicated()` |
| Count how many duplicates exist | `df.duplicated().sum()` |
| Inspect duplicate entries | `df[df.duplicated()]` |
| Remove duplicate rows | `df.drop_duplicates()` |
| Keep first / last / none | `keep='first'`, `keep='last'`, `keep=False` |
| Deduplicate on selected columns | `subset=['col1', 'col2']` |
| Verify results after deduplication | shape comparison + `.duplicated().sum()` |

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np

print("pandas version:", pd.__version__)
print("numpy version :", np.__version__)

## Step 2 — Load the Dataset and Inject Duplicates

We load the air quality CSV and then **intentionally inject duplicate rows** to simulate the kind of messy data you encounter in real-world scenarios (e.g., duplicate sensor submissions, data pipeline reruns, manual data entry errors).

In [ ]:
# ── Load the base air quality dataset ─────────────────────────────────────
file_path = '../data/raw/air_quality_missing.csv'
df_base = pd.read_csv(file_path)

print(f"Base dataset shape: {df_base.shape}  →  {df_base.shape[0]} rows × {df_base.shape[1]} columns")
display(df_base.head())

In [ ]:
# ── Inject duplicate rows to simulate real-world duplicates ───────────────
# Exact duplicates: repeat the first 10 rows as-is
exact_dupes = df_base.head(10).copy()

# Partial duplicates: same city+date but slightly different readings
partial_dupes = df_base.head(5).copy()
partial_dupes['pm25'] = partial_dupes['pm25'] + 0.1   # tiny rounding difference

# Combine to create working dataset
df = pd.concat([df_base, exact_dupes, partial_dupes], ignore_index=True)

print(f"Dataset WITH duplicates: {df.shape}  →  {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Rows added as exact duplicates  : {len(exact_dupes)}")
print(f"Rows added as partial duplicates: {len(partial_dupes)}")
display(df.tail(20))

---
## Step 3 — Understanding Duplicate Records

### What Are Duplicate Records?
A **duplicate record** is a row that repeats information already present in the dataset.

| Type | Description | Example |
|------|-------------|--------|
| **Exact duplicate** | Every column value matches | Same city, date, and all sensor readings |
| **Partial duplicate** | Key identifier columns match, minor values differ | Same city + date, slightly different PM2.5 |
| **Logical duplicate** | Different format, same meaning | 'Delhi' vs 'delhi', '2024-01-01' vs '01/01/2024' |

### Why Do Duplicates Occur?
- Sensor data sent twice (network retry)
- Manual data entry errors
- Multiple data pipelines appending the same source
- Merging datasets that share overlapping rows
- Copy-paste errors in spreadsheets

> **Rule:** Always detect duplicates before any analysis or aggregation.

In [ ]:
# ── What does a duplicate look like? ──────────────────────────────────────
# Show the first original row and its duplicate side-by-side
original_row  = df.iloc[0]
duplicate_row = df.iloc[df_base.shape[0]]   # first injected exact duplicate

comparison = pd.DataFrame({
    'Original Row (index 0)'     : original_row,
    'Duplicate Row (injected)'   : duplicate_row
})

print("Side-by-side comparison — Original vs Exact Duplicate:")
display(comparison)
print("\nAre they identical?", original_row.equals(duplicate_row))

---
## Step 4 — Detecting Duplicate Rows

**`df.duplicated()`** returns a boolean Series — `True` for every row that is a duplicate of an earlier row.

```python
df.duplicated()                         # all columns, keep first
df.duplicated(keep='last')              # mark all but the last occurrence
df.duplicated(keep=False)               # mark ALL duplicated rows (including original)
df.duplicated(subset=['city', 'date'])  # check only specific columns
```

> Detection must come **before** removal.

In [ ]:
# ── 4a. How many duplicate rows exist? ────────────────────────────────────
dup_mask = df.duplicated()   # True for rows that are duplicates of earlier rows

print(f"Total rows in dataset        : {len(df)}")
print(f"Exact duplicate rows detected: {dup_mask.sum()}")
print(f"Percentage of dataset        : {dup_mask.sum() / len(df) * 100:.2f}%")
print(f"Unique rows                  : {len(df) - dup_mask.sum()}")

In [ ]:
# ── 4b. Inspect the duplicate rows ────────────────────────────────────────
duplicated_rows = df[df.duplicated(keep=False)]  # keep=False → shows ALL occurrences

print(f"Rows involved in duplication (all occurrences): {len(duplicated_rows)}")
print()
display(duplicated_rows.sort_values(['city', 'date']).head(20))

In [ ]:
# ── 4c. Detect duplicates across specific columns only (partial duplicates) ─
# Same city + date = partial duplicate (regardless of sensor values)
partial_dup_mask = df.duplicated(subset=['city', 'date'])

print(f"Rows with duplicate city+date combinations: {partial_dup_mask.sum()}")
print()
print("Sample partial duplicates (same city & date, check values):")
display(df[df.duplicated(subset=['city', 'date'], keep=False)]
          .sort_values(['city', 'date'])
          .head(20))

In [ ]:
# ── 4d. Duplicate counts per city/station ─────────────────────────────────
dup_rows = df[df.duplicated(keep=False)]

if len(dup_rows) > 0:
    dup_by_city = (dup_rows
                   .groupby('city')
                   .size()
                   .reset_index(name='duplicate_count')
                   .sort_values('duplicate_count', ascending=False))
    print("Duplicate row counts by city:")
    display(dup_by_city)
else:
    print("No duplicates found.")

In [ ]:
# ── 4e. Summary detection table ───────────────────────────────────────────
detection_summary = pd.DataFrame({
    'Detection Method'     : [
        'All columns (keep first)',
        'All columns (keep last)',
        'All columns (keep=False)',
        'Subset: city + date'
    ],
    'Duplicates Found': [
        df.duplicated(keep='first').sum(),
        df.duplicated(keep='last').sum(),
        df.duplicated(keep=False).sum(),
        df.duplicated(subset=['city', 'date']).sum()
    ]
})

print("Duplicate Detection Summary:")
display(detection_summary)

---
## Step 5 — Removing Duplicate Records

**`df.drop_duplicates()`** removes duplicate rows from the DataFrame.

| Parameter | Effect |
|-----------|--------|
| `keep='first'` (default) | Keep first occurrence, drop subsequent duplicates |
| `keep='last'` | Keep last occurrence, drop earlier ones |
| `keep=False` | Drop ALL rows that have any duplicate (most aggressive) |
| `subset=['col']` | Only consider selected columns when detecting duplicates |
| `inplace=True` | Modify the original DataFrame directly |

> **Rule:** Removal should be intentional. Always know **which** duplicate you're keeping and **why**.

In [ ]:
# ── 5a. Remove duplicates — keep FIRST occurrence (default) ───────────────
df_dedup_first = df.drop_duplicates(keep='first')

print(f"Shape BEFORE: {df.shape}")
print(f"Shape AFTER  (keep='first'): {df_dedup_first.shape}")
print(f"Rows removed: {df.shape[0] - df_dedup_first.shape[0]}")
print(f"Duplicates remaining: {df_dedup_first.duplicated().sum()}")
display(df_dedup_first.head(10))

In [ ]:
# ── 5b. Remove duplicates — keep LAST occurrence ──────────────────────────
df_dedup_last = df.drop_duplicates(keep='last')

print(f"Shape BEFORE: {df.shape}")
print(f"Shape AFTER  (keep='last'): {df_dedup_last.shape}")
print(f"Rows removed: {df.shape[0] - df_dedup_last.shape[0]}")
print(f"Duplicates remaining: {df_dedup_last.duplicated().sum()}")

In [ ]:
# ── 5c. Remove ALL rows that have any duplicate (keep=False) ──────────────
# Useful when you cannot trust any version of a duplicated record
df_dedup_none = df.drop_duplicates(keep=False)

print(f"Shape BEFORE : {df.shape}")
print(f"Shape AFTER  (keep=False): {df_dedup_none.shape}")
print(f"Rows removed : {df.shape[0] - df_dedup_none.shape[0]}")
print(f"Duplicates remaining: {df_dedup_none.duplicated().sum()}")
print()
print("Note: keep=False is most aggressive — it removes even the original row.")

In [ ]:
# ── 5d. Remove partial duplicates — deduplicate on city + date only ────────
# When two rows share the same city+date, keep the one with HIGHER AQI reading
df_sorted = df.sort_values('aqi', ascending=False)   # highest AQI first
df_dedup_subset = df_sorted.drop_duplicates(subset=['city', 'date'], keep='first')
df_dedup_subset = df_dedup_subset.sort_values(['city', 'date']).reset_index(drop=True)

print(f"Shape BEFORE (partial dedup): {df.shape}")
print(f"Shape AFTER  (city+date dedup): {df_dedup_subset.shape}")
print(f"Rows removed: {df.shape[0] - df_dedup_subset.shape[0]}")
print(f"Remaining city+date duplicates: {df_dedup_subset.duplicated(subset=['city','date']).sum()}")
display(df_dedup_subset.head(10))

In [ ]:
# ── 5e. Reset index after deduplication ───────────────────────────────────
# Always reset index after dropping rows so the index is continuous
df_clean = df.drop_duplicates(keep='first').reset_index(drop=True)

print(f"Clean DataFrame shape: {df_clean.shape}")
print(f"Index range: {df_clean.index.min()} to {df_clean.index.max()}")
print(f"Index is continuous: {list(df_clean.index) == list(range(len(df_clean)))}")
display(df_clean.head())

---
## Step 6 — Verifying Deduplication Results

After removing duplicates, **always verify** the result.

Verification involves:
1. Comparing shape before vs after
2. Re-running `duplicated().sum()` to confirm zero duplicates
3. Checking that key records are preserved
4. Comparing key statistics (mean, count) to ensure the data isn't distorted

> **Rule:** Trust but verify. Always recheck after cleaning.

In [ ]:
# ── 6a. Shape comparison before vs after ──────────────────────────────────
print("Shape Comparison:")
print(f"  BEFORE deduplication : {df.shape}  ({df.shape[0]} rows)")
print(f"  AFTER  deduplication : {df_clean.shape}  ({df_clean.shape[0]} rows)")
print(f"  Rows removed         : {df.shape[0] - df_clean.shape[0]}")
print(f"  Data retained        : {df_clean.shape[0] / df.shape[0] * 100:.1f}%")

In [ ]:
# ── 6b. Recheck for remaining duplicates ──────────────────────────────────
remaining_exact  = df_clean.duplicated().sum()
remaining_partial = df_clean.duplicated(subset=['city', 'date']).sum()

print(f"Remaining exact duplicates        : {remaining_exact}")
print(f"Remaining city+date duplicates    : {remaining_partial}")

if remaining_exact == 0:
    print("✓  No exact duplicates remain — deduplication successful!")
else:
    print("✗  Duplicates still detected — review the deduplication step.")

In [ ]:
# ── 6c. Compare statistics before and after ───────────────────────────────
numeric_cols = ['pm25', 'pm10', 'no2', 'co', 'aqi', 'temperature', 'humidity']

stats_before = df[numeric_cols].describe().loc[['count', 'mean', 'std', 'min', 'max']]
stats_after  = df_clean[numeric_cols].describe().loc[['count', 'mean', 'std', 'min', 'max']]

print("=== Statistics BEFORE deduplication ===")
display(stats_before.round(2))

print("\n=== Statistics AFTER deduplication ===")
display(stats_after.round(2))

print("\nNote: Means should be similar — duplicates inflate counts, not distributions.")

In [ ]:
# ── 6d. Full deduplication summary table ──────────────────────────────────
summary = pd.DataFrame({
    'Strategy': [
        'Original (with dupes)',
        'drop_duplicates(keep="first")',
        'drop_duplicates(keep="last")',
        'drop_duplicates(keep=False)',
        'drop_duplicates(subset=[city,date])'
    ],
    'Rows': [
        df.shape[0],
        df.drop_duplicates(keep='first').shape[0],
        df.drop_duplicates(keep='last').shape[0],
        df.drop_duplicates(keep=False).shape[0],
        df.drop_duplicates(subset=['city', 'date']).shape[0]
    ],
    'Rows Removed': [
        0,
        df.shape[0] - df.drop_duplicates(keep='first').shape[0],
        df.shape[0] - df.drop_duplicates(keep='last').shape[0],
        df.shape[0] - df.drop_duplicates(keep=False).shape[0],
        df.shape[0] - df.drop_duplicates(subset=['city', 'date']).shape[0]
    ],
    'Duplicates After': [
        df.duplicated().sum(),
        0, 0, 0,
        df.drop_duplicates(subset=['city', 'date']).duplicated(subset=['city','date']).sum()
    ]
})

print("Deduplication Strategy Comparison:")
display(summary)

In [ ]:
# ── 6e. Final clean DataFrame info ────────────────────────────────────────
print("Final Clean DataFrame Info:")
print(f"  Shape   : {df_clean.shape}")
print(f"  Duplicates: {df_clean.duplicated().sum()}")
print(f"  Cities  : {df_clean['city'].nunique()} unique")
print(f"  Dates   : {df_clean['date'].nunique()} unique")
print()
df_clean.info()

---
## Summary

### Key Takeaways

| Step | What We Did | Key Method |
|------|-------------|------------|
| 1 | Loaded dataset and injected realistic duplicates | `pd.concat()` |
| 2 | Understood what exact and partial duplicates look like | Conceptual |
| 3 | Detected exact duplicates across all columns | `df.duplicated()` |
| 4 | Detected partial duplicates on city + date | `df.duplicated(subset=[...])` |
| 5 | Removed duplicates keeping first/last/none | `df.drop_duplicates(keep=...)` |
| 6 | Removed partial duplicates using subset | `df.drop_duplicates(subset=[...])` |
| 7 | Verified results — shape, count, statistics | Shape comparison + `.duplicated().sum()` |

### When to Use Each Strategy
| Situation | Strategy |
|-----------|----------|
| Data logged twice (same event) | `keep='first'` |
| Latest submission should win | `keep='last'` |
| Cannot trust any copy | `keep=False` |
| Same key, different values | Deduplicate on subset |

> **Remember:** Clean data leads to accurate analysis. Deduplication is not optional — it is essential.